# AR Rehabilitation Digital Pet RL Simulation
## Spatial Target Placement for Stroke Patients with Unilateral Spatial Neglect (USN)

### Research Overview
Unilateral Spatial Neglect (USN) is a frequent cognitive deficit following right-hemisphere stroke, leading patients to ignore objects and space on their contralateral (left) side. In this Augmented Reality (AR) rehabilitation system, a single virtual digital pet is placed in a 10m x 10m room ($20 \times 20$ grid = 400 exploration cells).

**Core Concept**: The RL agent does **not** control the patient's movement. Instead, the RL agent decides **WHERE the digital pet should appear** to maximize:
- Total room exploration
- Neglected-side (left) exploration depth
- Rehabilitation effectiveness
- Patient engagement without causing exhaustion or unearned immediate success.

### 1. Setup & Package Imports

In [ ]:
# Install dependencies if running on Kaggle or Google Colab
# !pip install gymnasium stable-baselines3 shimmy matplotlib plotly pandas numpy torch

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import gymnasium as gym
from stable_baselines3 import PPO, DQN, A2C

from config import ROOM_WIDTH, ROOM_HEIGHT, GRID_COLS, GRID_ROWS, OUTPUT_DIR
from environment import ARRehabEnv
from patient import PatientSim
from digital_pet import DigitalPet
from utils import set_seeds, HeuristicPolicies, calculate_metrics
from visualization import plot_room_and_trajectory, plot_exploration_heatmap, plot_learning_curves, plot_evaluation_comparison

set_seeds(42)
print("Environment and libraries successfully initialized!")

### 2. Environment & Patient Motion Model Demonstration
We test creating a randomized 10m x 10m room with obstacles (chairs, tables, plants, books) and simulating a patient step loop under severe USN spatial neglect.

In [ ]:
env = ARRehabEnv(neglect_severity=0.7, seed=42)
obs, info = env.reset()

# Choose pet position at extreme left cell (cell 182 -> x ~ 1.5, y ~ 4.5)
pet_action = 182
done = False
total_reward = 0.0

while not done:
    obs, reward, terminated, truncated, step_info = env.step(pet_action)
    total_reward += reward
    done = terminated or truncated

print(f"Demo Episode Finished in {env.step_count} steps!")
print(f"Total Reward: {total_reward:.2f}")
print(f"Left Side Exploration: {step_info['left_expl_ratio']*100:.1f}%")
print(f"Pet Found: {step_info['pet_found']}")

# Plot room trajectory map
img_path = plot_room_and_trajectory(env, env.patient, env.pet, title="Demo Episode - Patient Trajectory", save_name="demo_trajectory.png")
heatmap_path = plot_exploration_heatmap(env.patient.visited_grid, save_name="demo_heatmap.png")

from IPython.display import Image, display
display(Image(filename=img_path))
display(Image(filename=heatmap_path))

### 3. Reinforcement Learning Training (PPO, DQN, A2C)
We train PPO, DQN, and A2C agents using Stable-Baselines3 to learn the optimal pet position strategy.

In [ ]:
from train import train_all_algorithms

# Train all algorithms for 30,000 steps each
models, train_results = train_all_algorithms(total_timesteps=30000, seed=42)

### 4. Policy Benchmarking & Evaluation
We benchmark baseline heuristic policies against our trained RL policies over 50 test episodes.

In [ ]:
from evaluate import run_benchmark

eval_df = run_benchmark(num_episodes=50, seed=42)
display(eval_df)

### 5. Display Evaluation Charts & Visualizations

In [ ]:
# Display Learning Curves & Benchmark Bar Chart
curves_img = os.path.join(OUTPUT_DIR, "learning_curves_comparison.png")
bench_img = os.path.join(OUTPUT_DIR, "eval_benchmark_comparison.png")
ppo_traj_img = os.path.join(OUTPUT_DIR, "ppo_sample_trajectory.png")

if os.path.exists(curves_img):
    display(Image(filename=curves_img))
if os.path.exists(bench_img):
    display(Image(filename=bench_img))
if os.path.exists(ppo_traj_img):
    display(Image(filename=ppo_traj_img))